<a href="https://colab.research.google.com/github/mustafaabhatie/genesis_fn_liar_pipeline/blob/main/Genesis_FN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install & Import Dependencies

In [1]:

# Install required packages (if needed)
!pip install transformers nltk scikit-learn scipy torch ipywidgets

# Imports
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import shuffle
import torch
from transformers import BertTokenizer, BertModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from ipywidgets import FileUpload
from IPython.display import display
warnings.filterwarnings('ignore')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.8 MB/s eta 0:00:00


Dataset Upload

In [16]:

from ipywidgets import FileUpload
from IPython.display import display

upload = FileUpload(accept='.tsv', multiple=False)
display(upload)

def save_uploaded_file(upload_widget):
    if not upload_widget.value:
        print("No file uploaded yet!")
        return None
    for filename, fileinfo in upload_widget.value.items():
        with open(filename, 'wb') as f:
            f.write(fileinfo['content'])
        print(f"File saved as: {filename}")
    return filename




FileUpload(value={}, accept='.tsv', description='Upload')

No file uploaded yet!
Data path: None


In [17]:
# After uploading, run this:
data_path = save_uploaded_file(upload)
print("Data path:", data_path)

File saved as: train.tsv
Data path: train.tsv


Initialize NLTK Resources

In [18]:

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

Constants

In [19]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


# Dataset paths
DATA_FILE = "train.tsv"  # LIAR dataset
BERT_MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "./genesis_fn_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GA Layer 1: Feature Selection Parameters
GA1_POPULATION_SIZE = 50       # Population size for feature selection GA
GA1_GENERATIONS = 30          # Number of generations
GA1_CROSSOVER_RATE = 0.8      # Probability of crossover
GA1_MUTATION_RATE = 0.05      # Probability of mutation per gene
GA1_TOURNAMENT_SIZE = 3       # Tournament size for selection
GA1_ELITISM_COUNT = 2         # Number of best individuals preserved
GA1_INIT_SELECTION_RATE = 0.3 # Initial feature selection rate

# GA Layer 2: SVM Hyperparameter Optimization
GA2_POPULATION_SIZE = 30      # Population size for hyperparameter GA
GA2_GENERATIONS = 20          # Number of generations
GA2_CROSSOVER_RATE = 0.7      # Crossover probability
GA2_MUTATION_RATE = 0.1       # Mutation probability
GA2_ELITISM_COUNT = 2         # Elitism count
GA2_K_FOLDS = 3               # Cross-validation folds for fitness evaluation

# SVM Hyperparameter Ranges (Paper values)
SVM_C_MIN, SVM_C_MAX = 0.1, 10.0           # Regularization parameter C
SVM_GAMMA_MIN, SVM_GAMMA_MAX = 0.001, 1.0  # RBF kernel gamma
SVM_KERNEL_TYPES = ['linear', 'rbf']       # Kernel types to optimize

# Fitness Function Weights (Balancing performance vs complexity)
F1_WEIGHT = 0.35              # Weight for F1 score (primary metric)
ACCURACY_WEIGHT = 0.20        # Weight for accuracy
PRECISION_WEIGHT = 0.15       # Weight for precision
RECALL_WEIGHT = 0.15          # Weight for recall
COMPLEXITY_WEIGHT = 0.15      # Weight for feature count penalty

# Feature Selection Penalty Parameters
PENALTY_EXPONENT = 0.5        # Exponent for feature count penalty (sqrt)
MIN_FEATURES = 5              # Minimum features to avoid empty selection
MAX_FEATURE_PENALTY = 0.3     # Maximum penalty for too many features

Define Classes & Functions

In [20]:


# Genesis-FN Pipeline for Notebook Execution

import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from scipy.sparse import hstack, csr_matrix, issparse
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import torch
from transformers import BertTokenizer, BertModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

warnings.filterwarnings('ignore')


# Configuration Constants

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

GA1_POPULATION_SIZE = 50
GA1_GENERATIONS = 30
GA1_CROSSOVER_RATE = 0.8
GA1_MUTATION_RATE = 0.05
GA1_TOURNAMENT_SIZE = 3
GA1_ELITISM_COUNT = 2
MIN_FEATURES = 5

GA2_POPULATION_SIZE = 30
GA2_GENERATIONS = 20
GA2_CROSSOVER_RATE = 0.7
GA2_MUTATION_RATE = 0.1
GA2_ELITISM_COUNT = 2
GA2_K_FOLDS = 3

SVM_C_MIN, SVM_C_MAX = 0.1, 10.0
SVM_GAMMA_MIN, SVM_GAMMA_MAX = 0.001, 1.0
SVM_KERNEL_TYPES = ['linear', 'rbf']

F1_WEIGHT = 0.35
ACCURACY_WEIGHT = 0.20
PRECISION_WEIGHT = 0.15
RECALL_WEIGHT = 0.15
COMPLEXITY_WEIGHT = 0.15
PENALTY_EXPONENT = 0.5
MAX_FEATURE_PENALTY = 0.3

BERT_MODEL_NAME = "bert-base-uncased"


# Utility Functions

def initialize_nltk_resources():
    resources = ['stopwords', 'wordnet', 'punkt', 'averaged_perceptron_tagger']
    for resource in resources:
        try:
            nltk.data.find(f'tokenizers/{resource}' if resource == 'punkt' else f'corpora/{resource}')
        except LookupError:
            nltk.download(resource, quiet=True)

def load_and_prepare_data(filepath: str) -> Tuple[pd.DataFrame, pd.Series]:
    columns = [
        'id', 'label', 'statement', 'subject', 'speaker', 'job',
        'state', 'party', 'barely_true_c', 'false_c', 'half_true_c',
        'mostly_true_c', 'pants_on_fire_c', 'venue'
    ]
    df = pd.read_csv(filepath, sep='\t', names=columns)
    def binarize_label(label):
        if pd.isna(label):
            return 0
        label_str = str(label).lower().strip()
        return 1 if label_str in ['true', 'mostly-true', 'half-true'] else 0
    y = df['label'].apply(binarize_label)
    df = df[df['statement'].notna()]
    y = y[df.index]
    return df, y


# Text Preprocessor

class TextPreprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.stemmer = PorterStemmer()
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words.update({'breaking','exclusive','shocking','amazing','unbelievable','must','read','share','viral','secret','hidden','truth'})
    def preprocess(self, text: str) -> str:
        if not isinstance(text, str):
            text = str(text) if pd.notna(text) else ""
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        text = re.sub(r'@\w+|#\w+', '', text)
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\d+', ' ', text)
        tokens = text.split()
        tokens = [word for word in tokens if word not in self.stop_words]
        tokens = [self.stemmer.stem(word) for word in tokens]
        tokens = [self.lemmatizer.lemmatize(word) for word in tokens]
        tokens = [word for word in tokens if len(word) > 2]
        return ' '.join(tokens)


# Hybrid Feature Extractor

class HybridFeatureExtractor:
    def __init__(self, use_bert: bool = True, bert_batch_size: int = 16):
        self.use_bert = use_bert
        self.bert_batch_size = bert_batch_size
        self.tfidf_vectorizer = None
        self.onehot_encoder = None
        self.scaler = None
        self.bert_tokenizer = None
        self.bert_model = None
    def extract_tfidf_features(self, texts: List[str], max_features: int = 5000) -> csr_matrix:
        if self.tfidf_vectorizer is None:
            self.tfidf_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1,3), min_df=2, max_df=0.95, sublinear_tf=True, use_idf=True)
        return self.tfidf_vectorizer.fit_transform(texts)
    def extract_bert_embeddings(self, texts: List[str]) -> csr_matrix:
        if not self.use_bert:
            return csr_matrix((len(texts), 0))
        if self.bert_tokenizer is None or self.bert_model is None:
            self.bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
            self.bert_model = BertModel.from_pretrained(BERT_MODEL_NAME)
            self.bert_model.eval()
        embeddings = []
        with torch.no_grad():
            for i in range(0, len(texts), self.bert_batch_size):
                batch_texts = texts[i:i+self.bert_batch_size]
                inputs = self.bert_tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
                outputs = self.bert_model(**inputs)
                batch_embeddings = outputs.last_hidden_state.mean(dim=1).numpy()
                embeddings.append(batch_embeddings)
        X_bert = np.vstack(embeddings)
        if self.scaler is None:
            self.scaler = StandardScaler()
            X_bert = self.scaler.fit_transform(X_bert)
        else:
            X_bert = self.scaler.transform(X_bert)
        return csr_matrix(X_bert)
    def extract_metadata_features(self, df: pd.DataFrame) -> csr_matrix:
        categorical_cols = ['subject','speaker','job','state','party','venue']
        if self.onehot_encoder is None:
            self.onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True, max_categories=50)
            X_cat = self.onehot_encoder.fit_transform(df[categorical_cols])
        else:
            X_cat = self.onehot_encoder.transform(df[categorical_cols])
        numeric_cols = ['barely_true_c','false_c','half_true_c','mostly_true_c','pants_on_fire_c']
        X_num = df[numeric_cols].astype(float).values
        X_num = (X_num - X_num.mean(axis=0)) / (X_num.std(axis=0)+1e-8)
        return hstack([X_cat, csr_matrix(X_num)])
    def extract_readability_features(self, texts: List[str]) -> np.ndarray:
        scores = []
        for text in texts:
            words = text.split()
            sentences = re.split(r'[.!?]+', text)
            sentences = [s for s in sentences if s.strip()]
            if len(sentences)==0 or len(words)==0:
                scores.append([0,0,0]); continue
            avg_sentence_length = len(words)/len(sentences)
            avg_word_length = sum(len(w) for w in words)/len(words)
            unique_word_ratio = len(set(words))/len(words)
            scores.append([avg_sentence_length, avg_word_length, unique_word_ratio])
        return np.array(scores)
    def extract_all_features(self, df: pd.DataFrame, texts: List[str]) -> Tuple[csr_matrix, Dict]:
        X_tfidf = self.extract_tfidf_features(texts)
        X_bert = self.extract_bert_embeddings(texts)
        X_meta = self.extract_metadata_features(df)
        X_readability = csr_matrix(self.extract_readability_features(texts))
        X_hybrid = hstack([X_tfidf, X_bert, X_meta, X_readability])
        feature_metadata = {'tfidf_features':X_tfidf.shape[1],'bert_features':X_bert.shape[1],'metadata_features':X_meta.shape[1],'readability_features':X_readability.shape[1],'total_features':X_hybrid.shape[1]}
        return X_hybrid, feature_metadata


# FeatureSelectionGA

class FeatureSelectionGA:
    def __init__(self,pop_size=GA1_POPULATION_SIZE,generations=GA1_GENERATIONS,crossover_rate=GA1_CROSSOVER_RATE,mutation_rate=GA1_MUTATION_RATE,tournament_size=GA1_TOURNAMENT_SIZE,elitism_count=GA1_ELITISM_COUNT):
        self.pop_size=pop_size; self.generations=generations; self.crossover_rate=crossover_rate; self.mutation_rate=mutation_rate; self.tournament_size=tournament_size; self.elitism_count=elitism_count; self.fitness_cache={}
    def initialize_population(self,n_features:int)->np.ndarray:
        population=[]
        for i in range(self.pop_size):
            if i<self.pop_size//3: rate=random.uniform(0.1,0.3)
            elif i<2*self.pop_size//3: rate=random.uniform(0.3,0.6)
            else: rate=random.uniform(0.6,0.9)
            chromosome=(np.random.rand(n_features)<rate).astype(int)
            population.append(chromosome)
        return np.array(population)
    def calculate_fitness(self,chromosome,X,y,cache_key=None)->float:
        if cache_key and cache_key in self.fitness_cache: return self.fitness_cache[cache_key]
        selected_idx=np.where(chromosome)[0]; n_selected=len(selected_idx); n_total=X.shape[1]
        if n_selected<MIN_FEATURES: return 0.0
        X_selected=X[:,selected_idx]
        try:
            svm=SVC(kernel='linear',C=1.0,random_state=RANDOM_SEED)
            cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=RANDOM_SEED)
            f1=np.mean(cross_val_score(svm,X_selected,y,cv=cv,scoring='f1'))
            acc=np.mean(cross_val_score(svm,X_selected,y,cv=cv,scoring='accuracy'))
            prec=np.mean(cross_val_score(svm,X_selected,y,cv=cv,scoring='precision'))
            rec=np.mean(cross_val_score(svm,X_selected,y,cv=cv,scoring='recall'))
            penalty=min((n_selected**PENALTY_EXPONENT)/(n_total**PENALTY_EXPONENT),MAX_FEATURE_PENALTY)
            fitness=max(F1_WEIGHT*f1+ACCURACY_WEIGHT*acc+PRECISION_WEIGHT*prec+RECALL_WEIGHT*rec-COMPLEXITY_WEIGHT*penalty,0.0)
        except: fitness=0.0
        if cache_key: self.fitness_cache[cache_key]=fitness
        return fitness
    def tournament_selection(self,population,fitness)->np.ndarray:
        selected=[]
        for _ in range(self.pop_size):
            participants=np.random.choice(len(population),self.tournament_size,replace=False)
            winner_idx=participants[np.argmax(fitness[participants])]
            selected.append(population[winner_idx])
        return np.array(selected)
    def uniform_crossover(self,p1,p2)->Tuple[np.ndarray,np.ndarray]:
        if random.random()>self.crossover_rate: return p1.copy(),p2.copy()
        mask=np.random.rand(len(p1))>0.5
        return np.where(mask,p1,p2),np.where(mask,p2,p1)
    def adaptive_mutation(self,chromosome,generation,max_generation)->np.ndarray:
        mutated=chromosome.copy()
        rate=max(self.mutation_rate*(1.0-generation/max_generation),0.01)
        for i in range(len(chromosome)):
            if random.random()<rate: mutated[i]=1-mutated[i]
        return mutated
    def run(self,X,y)->Tuple[np.ndarray,Dict]:
        n_features=X.shape[1]; population=self.initialize_population(n_features)
        best_fitness=-np.inf; best_chromosome=None; history=[]
        for gen in range(self.generations):
            fitness_scores=np.array([self.calculate_fitness(ch,X,y,f"{gen}_{i}") for i,ch in enumerate(population)])
            gen_best_idx=np.argmax(fitness_scores)
            if fitness_scores[gen_best_idx]>best_fitness: best_fitness=fitness_scores[gen_best_idx]; best_chromosome=population[gen_best_idx].copy()
            history.append(best_fitness)
            parents=self.tournament_selection(population,fitness_scores)
            new_pop=[]
            elite_idx=np.argsort(fitness_scores)[-self.elitism_count:]
            for idx in elite_idx: new_pop.append(population[idx].copy())
            while len(new_pop)<self.pop_size:
                i1,i2=np.random.choice(len(parents),2,replace=False)
                c1,c2=self.uniform_crossover(parents[i1],parents[i2])
                new_pop.extend([self.adaptive_mutation(c1,gen,self.generations),self.adaptive_mutation(c2,gen,self.generations)])
            population=np.array(new_pop[:self.pop_size])
        selected_idx=np.where(best_chromosome)[0]
        return selected_idx,{'best_fitness':best_fitness,'fitness_history':history}

# ==============================
# SVMHyperparameterGA
# ==============================
class SVMHyperparameterGA:
    def __init__(self,pop_size=GA2_POPULATION_SIZE,generations=GA2_GENERATIONS,crossover_rate=GA2_CROSSOVER_RATE,mutation_rate=GA2_MUTATION_RATE,elitism_count=GA2_ELITISM_COUNT):
        self.pop_size=pop_size; self.generations=generations; self.crossover_rate=crossover_rate; self.mutation_rate=mutation_rate; self.elitism_count=elitism_count
    def encode(self,C,gamma,kernel)->np.ndarray:
        C_norm=(np.log10(C)-np.log10(SVM_C_MIN))/(np.log10(SVM_C_MAX)-np.log10(SVM_C_MIN))
        gamma_norm=(np.log10(gamma)-np.log10(SVM_GAMMA_MIN))/(np.log10(SVM_GAMMA_MAX)-np.log10(SVM_GAMMA_MIN))
        kernel_norm=0.0 if kernel=='linear' else 1.0
        return np.array([C_norm,gamma_norm,kernel_norm])
    def decode(self,ch)->Tuple[float,float,str]:
        C=10**(np.log10(SVM_C_MIN)+ch[0]*(np.log10(SVM_C_MAX)-np.log10(SVM_C_MIN)))
        gamma=10**(np.log10(SVM_GAMMA_MIN)+ch[1]*(np.log10(SVM_GAMMA_MAX)-np.log10(SVM_GAMMA_MIN)))
        kernel='linear' if ch[2]<0.5 else 'rbf'
        return C,gamma,kernel
    def initialize_population(self)->np.ndarray:
        pop=[]
        for _ in range(self.pop_size):
            C=10**np.random.uniform(np.log10(SVM_C_MIN),np.log10(SVM_C_MAX))
            gamma=10**np.random.uniform(np.log10(SVM_GAMMA_MIN),np.log10(SVM_GAMMA_MAX))
            kernel=random.choice(SVM_KERNEL_TYPES)
            pop.append(self.encode(C,gamma,kernel))
        return np.array(pop)
    def evaluate_fitness(self,ch,X,y)->float:
        C,gamma,kernel=self.decode(ch)
        try:
            svm=SVC(C=C,kernel=kernel,gamma=gamma if kernel=='rbf' else 'scale',random_state=RANDOM_SEED)
            cv=StratifiedKFold(n_splits=GA2_K_FOLDS,shuffle=True,random_state=RANDOM_SEED)
            return np.mean(cross_val_score(svm,X,y,cv=cv,scoring='f1'))
        except: return 0.0
    def crossover(self,p1,p2)->Tuple[np.ndarray,np.ndarray]:
        if random.random()>self.crossover_rate: return p1.copy(),p2.copy()
        alpha=random.random()
        return alpha*p1+(1-alpha)*p2,alpha*p2+(1-alpha)*p1
    def mutate(self,ch,generation)->np.ndarray:
        mutated=ch.copy()
        strength=max(0.1*(1-generation/self.generations),0.01)
        for i in range(len(ch)):
            if random.random()<self.mutation_rate:
                mutated[i]=np.clip(mutated[i]+np.random.normal(0,strength),0,1)
        return mutated
    def run(self,X,y)->Dict:
        pop=self.initialize_population(); best_fitness=-np.inf; best_ch=None
        for gen in range(self.generations):
            fitness_scores=np.array([self.evaluate_fitness(ch,X,y) for ch in pop])
            if fitness_scores.max()>best_fitness: best_fitness=fitness_scores.max(); best_ch=pop[np.argmax(fitness_scores)].copy()
            selected=[]
            for _ in range(self.pop_size):
                participants=np.random.choice(len(pop),3,replace=False)
                winner=participants[np.argmax(fitness_scores[participants])]
                selected.append(pop[winner])
            new_pop=[]
            elite_idx=np.argsort(fitness_scores)[-self.elitism_count:]
            for idx in elite_idx: new_pop.append(pop[idx].copy())
            while len(new_pop)<self.pop_size:
                i1,i2=np.random.choice(len(selected),2,replace=False)


FNPipeline

In [21]:

# ==============================
# GenesisFNPipeline Class
# ==============================
class GenesisFNPipeline:
    def __init__(self, use_bert=True, save_artifacts=False, verbose=True):
        self.use_bert = use_bert
        self.save_artifacts = save_artifacts
        self.verbose = verbose
        self.text_preprocessor = TextPreprocessor()
        self.feature_extractor = HybridFeatureExtractor(use_bert=use_bert)
        self.feature_selector = FeatureSelectionGA()
        self.hyperparam_optimizer = SVMHyperparameterGA()
        self.results = {}
        self.selected_features = None
        self.best_hyperparams = None
        self.final_model = None
        self.feature_metadata = None

    def log(self, message, level="INFO"):
        if self.verbose:
            print(f"[{level}] {message}")

    def run(self, data_path):
        self.log("="*70)
        self.log("GENESIS-FN: Evolutionary Fake News Detection Pipeline")
        self.log("="*70)

        # Step 1: Load data
        self.log("\n1. Loading and preparing data...")
        df, y = load_and_prepare_data(data_path)
        self.log(f"Loaded {len(df)} samples")

        # Step 2: Preprocess text
        self.log("\n2. Preprocessing text...")
        texts = df['statement'].apply(self.text_preprocessor.preprocess)

        # Step 3: Extract features
        self.log("\n3. Extracting hybrid features...")
        X_hybrid, self.feature_metadata = self.feature_extractor.extract_all_features(df, texts.tolist())
        self.log(f"Total features: {self.feature_metadata['total_features']}")

        # Step 4: Split data
        self.log("\n4. Splitting data...")
        X_temp, X_test, y_temp, y_test = train_test_split(X_hybrid, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED, stratify=y_temp)

        # Step 5: GA Layer 1 - Feature Selection
        self.log("\n5. Running GA for Feature Selection...")
        selected_indices, fs_stats = self.feature_selector.run(X_train, y_train)
        self.selected_features = selected_indices
        X_train_sel = X_train[:, selected_indices]
        X_val_sel = X_val[:, selected_indices]
        X_test_sel = X_test[:, selected_indices]

        # Step 6: GA Layer 2 - Hyperparameter Optimization
        self.log("\n6. Running GA for Hyperparameter Optimization...")
        hp_stats = self.hyperparam_optimizer.run(X_train_sel, y_train)
        self.best_hyperparams = {'C': hp_stats['best_C'], 'gamma': hp_stats['best_gamma'], 'kernel': hp_stats['best_kernel']}

        # Step 7: Train final model
        self.log("\n7. Training final model...")
        if self.best_hyperparams['kernel'] == 'linear':
            self.final_model = SVC(C=self.best_hyperparams['C'], kernel='linear', random_state=RANDOM_SEED, probability=True)
        else:
            self.final_model = SVC(C=self.best_hyperparams['C'], gamma=self.best_hyperparams['gamma'], kernel='rbf', random_state=RANDOM_SEED, probability=True)
        X_final_train = hstack([X_train_sel, X_val_sel])
        y_final_train = np.concatenate([y_train, y_val])
        self.final_model.fit(X_final_train, y_final_train)

        # Step 8: Evaluate
        self.log("\n8. Evaluating on test set...")
        y_pred = self.final_model.predict(X_test_sel)
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1_score': f1_score(y_test, y_pred)
        }
        self.results = {'model_performance': metrics, 'feature_selection': fs_stats, 'hyperparameter_optimization': hp_stats}
        return self.results

    def print_final_results(self):
        if not self.results:
            print("No results available. Run the pipeline first.")
            return
        metrics = self.results['model_performance']
        print("\nFINAL RESULTS:")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1-Score:  {metrics['f1_score']:.4f}")


Run the Pipeline

In [ ]:

# Initialize pipeline
pipeline = GenesisFNPipeline(use_bert=True, save_artifacts=False, verbose=True)

# Run pipeline on uploaded file
results = pipeline.run(data_path)

# Display final results
pipeline.print_final_results()


[INFO] ======================================================================
[INFO] GENESIS-FN: Evolutionary Fake News Detection Pipeline
[INFO] ======================================================================
[INFO] 
1. Loading and preparing data...
[INFO] Loaded 10240 samples
[INFO] 
2. Preprocessing text...
[INFO] 
3. Extracting hybrid features...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

[INFO] Total features: 6050
[INFO] 
4. Splitting data...
[INFO] 
5. Running GA for Feature Selection...
